In [1]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
import pandas as pd

# تحميل البيانات التي ولدناها سابقاً
df = pd.read_csv('car_market_data.csv')

# تحويل البيانات النوعية (Make, Fuel_Type) إلى بيانات رقمية (One-Hot Encoding)
df_encoded = pd.get_dummies(df, columns=['Make', 'Fuel_Type'])

# تحديد المتغيرات المستقلة (X) والهدف (y)
X = df_encoded.drop('Current_Market_Price', axis=1)
y = df_encoded['Current_Market_Price']

# تقسيم البيانات إلى مجموعة تدريب (80%) ومجموعة اختبار (20%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# إنشاء وتدريب النموذج
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# التنبؤ على بيانات الاختبار لتقييم الأداء
y_pred = model.predict(X_test)

# حساب المقاييس
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Mean Absolute Error (MAE): {mae:.2f}")
print(f"R-squared Score (R2): {r2:.4f}")

Mean Absolute Error (MAE): 4474.23
R-squared Score (R2): 0.9787


In [4]:
def evaluate_risk(make, year, mileage, fuel_type, purchase_price, X_train):
    # 1. إنشاء DataFrame لبيانات المدخلات
    input_data = pd.DataFrame({
        'Year': [year],
        'Mileage': [mileage],
        'Make_' + make: [1],
        'Fuel_Type_' + fuel_type: [1]
    })
    
    # 2. مطابقة الأعمدة (Reindexing)
    # هذه الخطوة هي الأهم: تضيف الأعمدة المفقودة بأصفار وتضمن الترتيب الصحيح
    input_data = input_data.reindex(columns=X_train.columns, fill_value=0)
    
    # 3. التنبؤ بالسعر
    predicted_price = model.predict(input_data)[0]
    
    # 4. محرك القرار
    margin = (predicted_price - purchase_price) / predicted_price
    
    print(f"السعر المتوقع في السوق: {predicted_price:.2f} درهم")
    
    if margin < -0.10:
        return "خطر عالي: سعر الشراء مبالغ فيه، يُنصح بطلب ضمانات إضافية."
    elif margin < 0:
        return "خطر متوسط: السعر قريب من السوق."
    else:
        return "عملية آمنة: السعر ضمن النطاق المقبول."

# استدعاء الدالة مع تمرير X_train التي استخدمناها في التدريب
print(evaluate_risk('Toyota', 2021, 100000, 'Diesel', 160000, X_train))

السعر المتوقع في السوق: 130522.43 درهم
خطر عالي: سعر الشراء مبالغ فيه، يُنصح بطلب ضمانات إضافية.


In [2]:
def stress_test_simulation(df, reduction_percentage):
    """
    محاكاة تأثير انخفاض أسعار السوق (أزمة اقتصادية مثلاً) على المحفظة.
    reduction_percentage: النسبة المئوية للانخفاض (مثلاً 0.20 لـ 20%)
    """
    simulated_df = df.copy()
    # تطبيق الانخفاض على الأسعار
    simulated_df['Market_Crash_Price'] = simulated_df['Current_Market_Price'] * (1 - reduction_percentage)
    
    # حساب الخسارة المحتملة
    loss = (df['Current_Market_Price'] - simulated_df['Market_Crash_Price']).mean()
    
    print(f"--- تقرير اختبار الضغط ---")
    print(f"في حال انخفاض السوق بنسبة {reduction_percentage*100}%:")
    print(f"متوسط الخسارة لكل سيارة في المحفظة: {loss:.2f} درهم")
    
    return simulated_df

# تشغيل المحاكاة (انخفاض 20%)
df_stressed = stress_test_simulation(df, 0.20)

--- تقرير اختبار الضغط ---
في حال انخفاض السوق بنسبة 20.0%:
متوسط الخسارة لكل سيارة في المحفظة: 23423.67 درهم
